# 05-9. 웹 접근 로그 분석 종합 실습 — 풀이·검증

## Goal

합성 fixture의 정상·오탐·인코딩·형식·의미 경계를 모두 처리하고, 마스킹된 조사 후보 산출물을 안전하게 저장한다.


## Setup

fixture에는 문서용 IP 대역, 반복 favicon 404, 승인 스캐너, 다수 고유 404 경로, 민감 경로 2xx, IPv6, CRLF, invalid UTF-8을 포함한다.


In [ ]:
from pathlib import Path
import sys


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "requirements.txt").is_file():
            return candidate
    raise FileNotFoundError("requirements.txt가 있는 저장소 루트에서 JupyterLab을 실행하세요.")


ROOT = find_project_root()
FIXTURE_DIR = ROOT / "fixtures" / "05-text-processing"

assert sys.version_info >= (3, 10)
assert FIXTURE_DIR.is_dir()

print("Python:", sys.version.split()[0])
print("실습 데이터:", FIXTURE_DIR)

from datetime import datetime, timezone
import os

LOG_PATHS = [
    FIXTURE_DIR / "web-access.log",
    FIXTURE_DIR / "web-access-invalid-utf8.log",
]
ALLOWED_OUTPUT_PARENT = Path(
    os.environ.get("CH05_ALLOWED_OUTPUT_PARENT", ROOT / "outputs")
)
OUTPUT_ROOT = Path(
    os.environ.get("CH05_OUTPUT_ROOT", ALLOWED_OUTPUT_PARENT / "web-log-analysis-lab")
)
RUN_ID = os.environ.get(
    "CH05_RUN_ID",
    datetime.now(timezone.utc).strftime("run-%Y%m%dT%H%M%S%fZ"),
)


## Steps

디코딩·파싱·검증·집계·가명처리·저장 함수를 정의한다.


In [ ]:
from __future__ import annotations

from collections import Counter, defaultdict
from datetime import datetime, timezone
from hashlib import sha256
from ipaddress import ip_address
from pathlib import Path
from tempfile import NamedTemporaryFile, mkdtemp
from urllib.parse import unquote, urlsplit
import hmac
import json
import os
import posixpath
import re
import shutil
import unicodedata

import numpy as np
import pandas as pd


class LogFormatError(ValueError):
    pass


class LogValidationError(ValueError):
    pass


COMBINED_PATTERN = re.compile(
    r'(?P<ip>\S+) \S+ \S+ '
    r'\[(?P<timestamp>[^\]]+)\] '
    r'"(?P<method>[A-Z]+) (?P<target>\S+) HTTP/(?P<http_version>[^"]+)" '
    r'(?P<status>[0-9]{3}) (?P<bytes>[0-9]+|-) '
    r'"(?P<referrer>[^"]*)" "(?P<user_agent>[^"]*)"'
)
INVALID_PERCENT = re.compile(r"%(?![0-9A-Fa-f]{2})")
SENSITIVE_SEGMENTS = {".env", ".git", "wp-admin", "phpmyadmin", "server-status"}
APPROVED_SCANNERS = {"198.51.100.77"}
RULE_VERSION = "chapter05-training-v2"
BATCH_SIZE = 25
ERROR_SAMPLE_LIMIT = 20
MAX_LINE_BYTES = 16_384
TRAINING_FIXTURE_SHA256 = {
    "web-access.log": "255cee6add742d2ddbc52cd1840ba5c99f1fe319b8b2e362e5a66a1dfc1a4aff",
    "web-access-invalid-utf8.log": "16ac8abb2f322704c0868b8036b7f6d6603de546cfeefb63b4cf3b91d8f963d1",
}


def sha256_file(path: Path) -> str:
    digest = sha256()
    with path.open("rb") as file:
        for chunk in iter(lambda: file.read(64 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def load_masking_key(paths: list[Path]) -> bytes:
    selected_paths = [path.resolve() for path in paths]
    if not selected_paths:
        raise RuntimeError("저장할 입력 경로가 없다.")
    if any(not path.is_file() for path in selected_paths):
        raise FileNotFoundError("저장 시점의 입력 경로를 확인해야 한다.")

    key_text = os.environ.get("LOG_MASKING_KEY")
    if key_text:
        key = key_text.encode("utf-8")
        if len(key) < 32:
            raise RuntimeError("LOG_MASKING_KEY는 UTF-8 기준 32바이트 이상이어야 한다.")
        return key

    training_paths = {
        (FIXTURE_DIR / name).resolve(): expected_digest
        for name, expected_digest in TRAINING_FIXTURE_SHA256.items()
    }
    if all(
        path in training_paths
        and hmac.compare_digest(sha256_file(path), training_paths[path])
        for path in selected_paths
    ):
        return b"chapter05-training-key-not-for-real-data"

    raise RuntimeError("실제 로그 결과를 저장하려면 LOG_MASKING_KEY가 필요하다.")


def remove_record_separator(raw_line: bytes) -> bytes:
    return raw_line.removesuffix(b"\n").removesuffix(b"\r")


def normalize_request_path(target: str) -> tuple[str, str, bool]:
    try:
        parts = urlsplit(target)
    except ValueError as exc:
        raise LogValidationError("URL 구조를 해석할 수 없다.") from exc
    if parts.scheme or parts.netloc:
        raise LogValidationError("절대 URL 형식을 허용하지 않는다.")
    raw_path = parts.path or "/"
    if not raw_path.startswith("/") or INVALID_PERCENT.search(raw_path):
        raise LogValidationError("요청 경로 형식이 올바르지 않다.")
    try:
        decoded = unquote(raw_path, encoding="utf-8", errors="strict")
    except UnicodeDecodeError as exc:
        raise LogValidationError("경로의 percent-encoding을 UTF-8로 해석할 수 없다.") from exc
    decoded = unicodedata.normalize("NFC", decoded).replace("\\", "/")
    if len(decoded) > 2048 or any(ord(char) < 32 or ord(char) == 127 for char in decoded):
        raise LogValidationError("경로 길이 또는 제어 문자 규칙을 위반했다.")
    traversal = any(segment == ".." for segment in decoded.split("/"))
    normalized = posixpath.normpath("/" + decoded.lstrip("/"))
    return raw_path, normalized, traversal


def user_agent_class(value: str) -> str:
    lowered = value.casefold()
    if "approvedtrainingscanner" in lowered:
        return "approved_scanner"
    if "scanner" in lowered:
        return "scanner"
    if "browser" in lowered or "mozilla" in lowered:
        return "browser"
    return "other"


def parse_combined_log(text: str) -> dict:
    match = COMBINED_PATTERN.fullmatch(text)
    if match is None:
        raise LogFormatError("Combined Log 형식과 일치하지 않는다.")
    fields = match.groupdict()
    try:
        canonical_ip = str(ip_address(fields["ip"]))
        timestamp = datetime.strptime(fields["timestamp"], "%d/%b/%Y:%H:%M:%S %z").astimezone(timezone.utc)
    except ValueError as exc:
        raise LogValidationError("IP 또는 timestamp 의미 검증에 실패했다.") from exc
    status = int(fields["status"])
    if not 100 <= status <= 599:
        raise LogValidationError("상태 코드는 100~599 범위여야 한다.")
    if fields["bytes"] == "-":
        response_bytes = None
    else:
        if len(fields["bytes"]) > 19:
            raise LogValidationError("응답 바이트 값이 허용 범위를 벗어났다.")
        response_bytes = int(fields["bytes"])
        if response_bytes > 2**63 - 1:
            raise LogValidationError("응답 바이트 값이 허용 범위를 벗어났다.")
    raw_path, normalized_path, traversal = normalize_request_path(fields["target"])
    segments = {segment.casefold() for segment in normalized_path.split("/") if segment}
    sensitive = bool(segments & SENSITIVE_SEGMENTS)
    return {
        "ip": canonical_ip,
        "timestamp": timestamp,
        "window": timestamp.replace(minute=(timestamp.minute // 5) * 5, second=0, microsecond=0),
        "method": fields["method"],
        "status": status,
        "response_bytes": response_bytes,
        "raw_path": raw_path,
        "normalized_path": normalized_path,
        "normalization_changed": raw_path != normalized_path,
        "path_traversal": traversal,
        "sensitive": sensitive,
        "user_agent_class": user_agent_class(fields["user_agent"]),
    }


def new_summary() -> dict:
    return {
        "total_lines": 0,
        "parsed_lines": 0,
        "encoding_errors": 0,
        "format_errors": 0,
        "oversized_line_errors": 0,
        "validation_errors": 0,
        "known_bytes_rows": 0,
        "missing_bytes": 0,
        "status": Counter(),
        "method": Counter(),
        "windows": defaultdict(lambda: {
            "total_requests": 0,
            "not_found_404": 0,
            "sensitive_requests": 0,
            "sensitive_2xx": 0,
            "normalization_changed_requests": 0,
            "path_traversal_requests": 0,
            "not_found_paths": set(),
            "sensitive_paths": set(),
        }),
        "error_samples": [],
        "_input_paths": (),
    }


def error_sample(
    source: str,
    line: int,
    length: int,
    error_type: str,
) -> dict:
    return {
        "source": source,
        "line": line,
        "type": error_type,
        "length": length,
    }


def merge_batch(summary: dict, records: list[dict]) -> None:
    if not records:
        return
    frame = pd.DataFrame.from_records(records)
    summary["parsed_lines"] += len(frame)
    summary["status"].update(frame["status"].value_counts().to_dict())
    summary["method"].update(frame["method"].value_counts().to_dict())
    summary["known_bytes_rows"] += int(frame["response_bytes"].notna().sum())
    summary["missing_bytes"] += int(frame["response_bytes"].isna().sum())

    for (source_ip, window), group in frame.groupby(["ip", "window"], sort=False):
        bucket = summary["windows"][(source_ip, window)]
        not_found = group["status"].eq(404)
        sensitive = group["sensitive"]
        bucket["total_requests"] += len(group)
        bucket["not_found_404"] += int(not_found.sum())
        bucket["sensitive_requests"] += int(sensitive.sum())
        bucket["sensitive_2xx"] += int((sensitive & group["status"].between(200, 299)).sum())
        bucket["normalization_changed_requests"] += int(group["normalization_changed"].sum())
        bucket["path_traversal_requests"] += int(group["path_traversal"].sum())
        bucket["not_found_paths"].update(group.loc[not_found, "normalized_path"])
        bucket["sensitive_paths"].update(group.loc[sensitive, "normalized_path"])


def analyze_files(paths: list[Path], batch_size: int = BATCH_SIZE) -> tuple[dict, list[dict]]:
    summary = new_summary()
    summary["_input_paths"] = tuple(str(path.resolve()) for path in paths)
    batch: list[dict] = []
    input_manifest = []

    for source_number, path in enumerate(paths, start=1):
        source_label = f"input-{source_number}"
        digest = sha256()
        with path.open("rb") as file:
            line_number = 0
            while True:
                raw_line = file.readline(MAX_LINE_BYTES + 1)
                if not raw_line:
                    break
                line_number += 1
                digest.update(raw_line)
                summary["total_lines"] += 1

                if len(raw_line) > MAX_LINE_BYTES:
                    oversized_length = len(raw_line)
                    while not raw_line.endswith(b"\n"):
                        raw_line = file.readline(MAX_LINE_BYTES + 1)
                        if not raw_line:
                            break
                        digest.update(raw_line)
                        oversized_length += len(raw_line)
                    summary["oversized_line_errors"] += 1
                    if len(summary["error_samples"]) < ERROR_SAMPLE_LIMIT:
                        summary["error_samples"].append(
                            error_sample(
                                source_label,
                                line_number,
                                oversized_length,
                                "line_too_long",
                            )
                        )
                    continue

                raw = remove_record_separator(raw_line)
                try:
                    text = raw.decode("utf-8", errors="strict")
                except UnicodeDecodeError:
                    summary["encoding_errors"] += 1
                    if len(summary["error_samples"]) < ERROR_SAMPLE_LIMIT:
                        summary["error_samples"].append(
                            error_sample(source_label, line_number, len(raw), "encoding")
                        )
                    continue
                try:
                    batch.append(parse_combined_log(text))
                except LogFormatError:
                    summary["format_errors"] += 1
                    if len(summary["error_samples"]) < ERROR_SAMPLE_LIMIT:
                        summary["error_samples"].append(
                            error_sample(source_label, line_number, len(raw), "format")
                        )
                except LogValidationError:
                    summary["validation_errors"] += 1
                    if len(summary["error_samples"]) < ERROR_SAMPLE_LIMIT:
                        summary["error_samples"].append(
                            error_sample(source_label, line_number, len(raw), "validation")
                        )
                if len(batch) >= batch_size:
                    merge_batch(summary, batch)
                    batch.clear()
        merge_batch(summary, batch)
        batch.clear()
        input_manifest.append({
            "name": path.name,
            "sha256": digest.hexdigest(),
            "size_bytes": path.stat().st_size,
        })
    return summary, input_manifest


def build_features(summary: dict) -> pd.DataFrame:
    base_columns = [
        "ip",
        "window",
        "total_requests",
        "not_found_404",
        "unique_404_paths",
        "sensitive_requests",
        "unique_sensitive_paths",
        "sensitive_2xx",
        "normalization_changed_requests",
        "path_traversal_requests",
        "approved_scanner_context",
    ]
    rows = []
    for (source_ip, window), values in summary["windows"].items():
        rows.append({
            "ip": source_ip,
            "window": window,
            "total_requests": values["total_requests"],
            "not_found_404": values["not_found_404"],
            "unique_404_paths": len(values["not_found_paths"]),
            "sensitive_requests": values["sensitive_requests"],
            "unique_sensitive_paths": len(values["sensitive_paths"]),
            "sensitive_2xx": values["sensitive_2xx"],
            "normalization_changed_requests": values["normalization_changed_requests"],
            "path_traversal_requests": values["path_traversal_requests"],
            "approved_scanner_context": source_ip in APPROVED_SCANNERS,
        })
    features = pd.DataFrame(rows, columns=base_columns)
    if features.empty:
        features["not_found_rate"] = pd.Series(dtype="float64")
        features["candidate"] = pd.Series(dtype="bool")
        features["reason"] = pd.Series(dtype="string")
        return features

    features = features.sort_values(["window", "ip"]).reset_index(drop=True)
    total = features["total_requests"].to_numpy(dtype=np.int64)
    not_found = features["not_found_404"].to_numpy(dtype=np.int64)
    not_found_rate = np.divide(
        not_found,
        total,
        out=np.zeros(total.shape, dtype=float),
        where=total > 0,
    )
    unique_404 = features["unique_404_paths"].to_numpy(dtype=np.int64)
    sensitive_requests = features["sensitive_requests"].to_numpy(dtype=np.int64)
    unique_sensitive = features["unique_sensitive_paths"].to_numpy(dtype=np.int64)
    sensitive_2xx = features["sensitive_2xx"].to_numpy(dtype=np.int64)
    scan_signal = (not_found >= 8) & (not_found_rate >= 0.70) & (unique_404 >= 6)
    sensitive_signal = (sensitive_2xx >= 1) | ((sensitive_requests >= 3) & (unique_sensitive >= 2))
    features["not_found_rate"] = not_found_rate
    features["candidate"] = scan_signal | sensitive_signal
    features["reason"] = np.select(
        [scan_signal & sensitive_signal, sensitive_signal, scan_signal],
        ["scan+sensitive", "sensitive", "scan"],
        default="-",
    )
    return features


def alias(value: str, prefix: str, masking_key: bytes) -> str:
    message = f"{prefix}:{value}".encode("utf-8")
    digest = hmac.new(masking_key, message, "sha256").hexdigest()[:24]
    return f"{prefix}_{digest}"


def masked_features(features: pd.DataFrame, masking_key: bytes) -> pd.DataFrame:
    output = features.drop(columns=["ip"]).copy()
    output.insert(
        0,
        "source_alias",
        features["ip"].map(lambda value: alias(value, "ip", masking_key)),
    )
    output["window"] = output["window"].map(lambda value: value.isoformat())
    return output


def validate_summary(summary: dict) -> None:
    errors = (
        summary["encoding_errors"]
        + summary["format_errors"]
        + summary["validation_errors"]
        + summary["oversized_line_errors"]
    )
    assert summary["total_lines"] == summary["parsed_lines"] + errors
    assert summary["parsed_lines"] == sum(summary["status"].values())
    assert summary["parsed_lines"] == sum(summary["method"].values())
    assert summary["parsed_lines"] == summary["known_bytes_rows"] + summary["missing_bytes"]


def atomic_write_text(path: Path, text: str, output_root: Path) -> None:
    root = output_root.resolve()
    target = path.resolve(strict=False)
    target.relative_to(root)
    if path.is_symlink():
        raise ValueError("심볼릭 출력을 허용하지 않는다.")
    temporary = None
    try:
        with NamedTemporaryFile(
            "w",
            encoding="utf-8",
            newline="",
            dir=path.parent,
            delete=False,
        ) as tmp:
            temporary = Path(tmp.name)
            tmp.write(text)
            tmp.flush()
            os.fsync(tmp.fileno())
        os.chmod(temporary, 0o600)
        os.replace(temporary, path)
    finally:
        if temporary is not None:
            temporary.unlink(missing_ok=True)


def save_outputs(
    summary: dict,
    features: pd.DataFrame,
    input_paths: list[Path],
    input_manifest: list[dict],
    output_root: Path,
    run_id: str,
    allowed_parent: Path,
) -> Path:
    # 저장 시점의 실제 입력 경로로 키 정책을 다시 검증한다.
    selected_paths = tuple(str(path.resolve()) for path in input_paths)
    if selected_paths != summary.get("_input_paths"):
        raise RuntimeError("분석한 입력과 저장 시점의 입력 경로가 다르다.")
    masking_key = load_masking_key(input_paths)
    allowed_parent.mkdir(parents=True, exist_ok=True)
    if allowed_parent.is_symlink():
        raise ValueError("심볼릭 허용 상위 경로를 허용하지 않는다.")
    allowed_parent = allowed_parent.resolve()

    if output_root.is_symlink():
        raise ValueError("심볼릭 출력 루트를 허용하지 않는다.")
    output_root.resolve(strict=False).relative_to(allowed_parent)
    output_root.mkdir(parents=True, exist_ok=True)
    output_root = output_root.resolve()
    if not re.fullmatch(r"[A-Za-z0-9_-]+", run_id):
        raise ValueError("잘못된 run_id이다.")
    run_dir = output_root / run_id
    if run_dir.exists() or run_dir.is_symlink():
        raise FileExistsError("동일한 run_id의 결과가 이미 존재한다.")

    masked = masked_features(features, masking_key)
    candidates = masked.loc[masked["candidate"]].copy()
    quality = {
        key: summary[key]
        for key in (
            "total_lines", "parsed_lines", "encoding_errors", "format_errors",
            "validation_errors", "oversized_line_errors", "known_bytes_rows",
            "missing_bytes",
        )
    }
    manifest = {
        "rule_version": RULE_VERSION,
        "inputs": input_manifest,
        "quality": quality,
        "thresholds": {
            "scan": {"not_found_404": 8, "not_found_rate": 0.70, "unique_404_paths": 6},
            "sensitive": {"sensitive_2xx": 1, "sensitive_requests": 3, "unique_sensitive_paths": 2},
        },
    }
    quality_frame = pd.DataFrame([quality])
    status_frame = pd.DataFrame(
        sorted(summary["status"].items()),
        columns=["status", "requests"],
    )
    staging_dir = Path(mkdtemp(prefix=f".staging-{run_id}-", dir=output_root))
    staging_dir.chmod(0o700)
    try:
        atomic_write_text(staging_dir / "window-features.csv", masked.to_csv(index=False), output_root)
        atomic_write_text(staging_dir / "triage-candidates.csv", candidates.to_csv(index=False), output_root)
        atomic_write_text(staging_dir / "quality-report.csv", quality_frame.to_csv(index=False), output_root)
        atomic_write_text(staging_dir / "status-counts.csv", status_frame.to_csv(index=False), output_root)
        atomic_write_text(
            staging_dir / "manifest.json",
            json.dumps(manifest, ensure_ascii=False, indent=2) + "\n",
            output_root,
        )
        os.replace(staging_dir, run_dir)
        return run_dir
    finally:
        if staging_dir.exists():
            shutil.rmtree(staging_dir, ignore_errors=True)


### 분석 실행

원문 오류 샘플 대신 오류 유형·길이·해시 접두사만 남긴다.


In [ ]:
summary, input_manifest = analyze_files(LOG_PATHS)
validate_summary(summary)
features = build_features(summary)
preview_masking_key = load_masking_key(LOG_PATHS)
masked = masked_features(features, preview_masking_key)

quality = {
    key: summary[key]
    for key in (
        "total_lines", "parsed_lines", "encoding_errors", "format_errors",
        "validation_errors", "oversized_line_errors", "known_bytes_rows",
        "missing_bytes",
    )
}
print(quality)
print(masked.loc[masked["candidate"]])
print(summary["error_samples"])


## Checks

품질 불변식과 합성 시나리오의 오탐·정탐 경계를 대조한다.


In [ ]:
assert quality["encoding_errors"] == 1
assert quality["format_errors"] == 1
assert quality["validation_errors"] == 3
assert quality["oversized_line_errors"] == 0
assert quality["missing_bytes"] == 1
assert quality["total_lines"] == 87
assert quality["parsed_lines"] == 82
assert quality["known_bytes_rows"] == 81
assert len(summary["error_samples"]) <= ERROR_SAMPLE_LIMIT
assert Counter(sample["type"] for sample in summary["error_samples"]) == {
    "encoding": 1,
    "format": 1,
    "validation": 3,
}

candidates = features.loc[features["candidate"]]
candidate_ips = set(candidates["ip"])
assert "198.51.100.40" in candidate_ips          # 다수 고유 404 경로
assert "203.0.113.50" in candidate_ips           # 민감 경로 2xx 1건
assert "198.51.100.77" in candidate_ips          # 승인 스캐너도 신호는 보존
assert "192.0.2.20" not in candidate_ips          # 동일 favicon 404 반복 오탐 억제
assert "192.0.2.10" not in candidate_ips          # 소량의 일반 404
assert int(features["normalization_changed_requests"].sum()) == 2
assert int(features["path_traversal_requests"].sum()) == 2

assert not any("raw" in sample for sample in summary["error_samples"])
assert all(
    set(sample) == {"source", "line", "type", "length"}
    for sample in summary["error_samples"]
)
assert "ip" not in masked.columns and "source_alias" in masked.columns
assert alias("same-value", "ip", preview_masking_key) != alias(
    "same-value", "path", preview_masking_key
)

empty_features = build_features(new_summary())
assert empty_features.empty and {"candidate", "reason"} <= set(empty_features.columns)

invalid_target = (
    '192.0.2.1 - - [14/Aug/2026:01:00:00 +0000] '
    '"GET http://[::1 HTTP/1.1" 200 10 "-" "TrainingBrowser/1.0"'
)
huge_bytes = (
    '192.0.2.1 - - [14/Aug/2026:01:00:00 +0000] '
    f'"GET / HTTP/1.1" 200 {"9" * 4_300} "-" "TrainingBrowser/1.0"'
)
for invalid_line in (invalid_target, huge_bytes):
    try:
        parse_combined_log(invalid_line)
    except LogValidationError:
        pass
    else:
        raise AssertionError("경계 입력을 검증 오류로 분류하지 않았다.")

non_ascii_digits = (
    '192.0.2.1 - - [14/Aug/2026:01:00:00 +0000] '
    '"GET / HTTP/1.1" ２００ １０ "-" "TrainingBrowser/1.0"'
)
try:
    parse_combined_log(non_ascii_digits)
except LogFormatError:
    pass
else:
    raise AssertionError("status·bytes에서 ASCII 숫자 경계를 지키지 않았다.")

ALLOWED_OUTPUT_PARENT.mkdir(parents=True, exist_ok=True)
valid_line = (
    '203.0.113.10 - - [14/Aug/2026:10:30:00 +0900] '
    '"GET /login HTTP/1.1" 200 443 "-" "TrainingBrowser/1.0"'
)
with NamedTemporaryFile("wb", dir=ALLOWED_OUTPUT_PARENT, delete=False) as temporary_file:
    oversized_path = Path(temporary_file.name)
    temporary_file.write(b"x" * (MAX_LINE_BYTES + 1) + b"\n")
    temporary_file.write(valid_line.encode("utf-8") + b"\n")
try:
    oversized_summary, _ = analyze_files([oversized_path])
    assert oversized_summary["total_lines"] == 2
    assert oversized_summary["format_errors"] == 0
    assert oversized_summary["oversized_line_errors"] == 1
    assert oversized_summary["parsed_lines"] == 1
    assert oversized_summary["error_samples"][0]["type"] == "line_too_long"
finally:
    oversized_path.unlink(missing_ok=True)
print("품질·탐지 경계 검증 통과")


### 안전한 산출물 저장

실제 로그에서는 가명처리 키를 환경변수·비밀 관리 시스템에서 제공하고 저장소에 넣지 않는다.


In [ ]:
from unittest.mock import patch

# 저장 시점에 현재 입력 경로를 다시 검증하는지 확인한다.
ALLOWED_OUTPUT_PARENT.mkdir(parents=True, exist_ok=True)
with NamedTemporaryFile("wb", dir=ALLOWED_OUTPUT_PARENT, delete=False) as temporary_input:
    outside_input = Path(temporary_input.name)
    temporary_input.write(b"synthetic non-fixture input\n")
previous_key = os.environ.pop("LOG_MASKING_KEY", None)
try:
    try:
        load_masking_key([outside_input])
    except RuntimeError:
        pass
    else:
        raise AssertionError("외부 입력에 교육용 키를 사용했다.")

    try:
        save_outputs(
            summary,
            features,
            [OUTPUT_ROOT / "not-a-training-fixture.log"],
            input_manifest,
            OUTPUT_ROOT,
            f"{RUN_ID}-key-rejected",
            ALLOWED_OUTPUT_PARENT,
        )
    except RuntimeError:
        pass
    else:
        raise AssertionError("실제 로그 키 정책을 우회했다.")
finally:
    if previous_key is not None:
        os.environ["LOG_MASKING_KEY"] = previous_key
    outside_input.unlink(missing_ok=True)

# 파일 저장 중 실패해도 완성 run과 staging 디렉터리가 남지 않는지 확인한다.
failure_run_id = f"{RUN_ID}-write-failure"
with patch("os.replace", side_effect=OSError("교육용 저장 실패")):
    try:
        save_outputs(
            summary,
            features,
            LOG_PATHS,
            input_manifest,
            OUTPUT_ROOT,
            failure_run_id,
            ALLOWED_OUTPUT_PARENT,
        )
    except OSError:
        pass
    else:
        raise AssertionError("저장 실패가 전파되지 않았다.")
assert not (OUTPUT_ROOT / failure_run_id).exists()
assert not list(OUTPUT_ROOT.glob(f".staging-{failure_run_id}-*"))

run_dir = save_outputs(
    summary,
    features,
    LOG_PATHS,
    input_manifest,
    OUTPUT_ROOT,
    RUN_ID,
    ALLOWED_OUTPUT_PARENT,
)
expected_files = {
    "window-features.csv",
    "triage-candidates.csv",
    "quality-report.csv",
    "status-counts.csv",
    "manifest.json",
}
assert {path.name for path in run_dir.iterdir()} == expected_files

candidate_text = (run_dir / "triage-candidates.csv").read_text(encoding="utf-8")
assert "198.51.100.40" not in candidate_text
assert "token=do-not-store" not in candidate_text
assert "raw_path" not in candidate_text
print("저장 위치:", run_dir)


## Next Steps

교육용 임계값은 보편적 침해 판정 기준이 아니다. 서비스 기준선, 프록시·NAT, 승인 스캐너, 자산 역할, 인증 이벤트를 함께 확인해 신호→조사→판정의 순서를 유지한다.
